# Adaptive-MCS Evaluation — Per-RX KPIs across TrafficClass profiles

Companion to `../traffic_class_eval/evaluation.ipynb` but for the NACK-rate
adaptive-MCS campaign.  All figures are computed from the 5 tidy CSVs that
`extract_kpis.py` writes under `results_traffic_class_adaptive_eval/tidy/`.

§6b ("Adaptive MCS behaviour") is the new section unique to this notebook.

The fixed-vs-adaptive side-by-side comparison lives in `comparison.ipynb`.


## §1. Setup


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS = Path("./results_traffic_class_adaptive_eval")
TIDY    = RESULTS / "tidy"

CLASS_ORDER  = ["periodic1", "periodic2", "periodic3", "aperiodic1", "aperiodic2"]
CLASS_COLORS = {
    "periodic1":  "#2c7bb6",
    "periodic2":  "#abd9e9",
    "periodic3":  "#d7191c",
    "aperiodic1": "#fdae61",
    "aperiodic2": "#7b3294",
}
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

rx = pd.read_csv(TIDY / "per_rx_kpis.csv")
ts = pd.read_csv(TIDY / "time_series.csv")
ru = pd.read_csv(TIDY / "resource_usage.csv")
hq = pd.read_csv(TIDY / "harq_stats.csv")
ad_path = TIDY / "adapt_log.csv"
ad = pd.read_csv(ad_path) if ad_path.is_file() else pd.DataFrame()
for d in (rx, ts, ru, hq, ad):
    if "trafficClass" in d.columns:
        d["trafficClass"] = pd.Categorical(d["trafficClass"], CLASS_ORDER, ordered=True)
print(f"rx={len(rx)} ts={len(ts)} ru={len(ru)} hq={len(hq)} adapt_log={len(ad)}")


## §2. Per-RX distributions of PRR / PIR / throughput


In [ ]:
classes = [c for c in CLASS_ORDER if c in rx["trafficClass"].unique()]
metrics = [("prr", "Group PRR", (0, 1.02)),
           ("pir_ms", "PIR [ms]", None),
           ("tput_kbps", "Throughput [kbps]", None)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (col, label, ylim) in zip(axes, metrics):
    sns.violinplot(data=rx, x="trafficClass", y=col, ax=ax, palette=CLASS_COLORS,
                   inner="quartile", cut=0, order=classes)
    sns.stripplot(data=rx, x="trafficClass", y=col, ax=ax, color="0.2",
                  size=2.5, alpha=0.5, order=classes)
    ax.set_title(label); ax.set_xlabel(""); ax.set_ylabel(label)
    if ylim: ax.set_ylim(ylim)
    for t in ax.get_xticklabels(): t.set_rotation(20)
fig.suptitle("Per-RX KPI distributions — Adaptive MCS", y=1.02)
fig.tight_layout(); plt.show()


## §3. KPI vs distance from TX


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (col, label, ylim) in zip(axes, metrics):
    for c in classes:
        sub = rx[rx["trafficClass"] == c]
        grp = sub.groupby("rx_node_id").agg(
            mean_d=("init_dist_m", "mean"),
            mean_v=(col, "mean"),
            std_v=(col, "std"),
        ).sort_values("mean_d")
        ax.errorbar(grp["mean_d"], grp["mean_v"], yerr=grp["std_v"],
                    fmt="o-", color=CLASS_COLORS[c], capsize=3, lw=1.5,
                    label=c, alpha=0.85)
    ax.set_xlabel("TX-RX initial distance [m]"); ax.set_ylabel(label)
    if ylim is not None: ax.set_ylim(ylim)
    ax.set_title(f"{label} vs distance"); ax.legend(fontsize=8)
fig.suptitle("Per-RX KPI vs distance — Adaptive MCS", y=1.02)
fig.tight_layout(); plt.show()


## §4. Time-series variation (1-s bins)


In [ ]:
rx_nodes = sorted(ts["rx_node_id"].unique())
fig, axes = plt.subplots(len(classes), 1, figsize=(11, 2.4 * len(classes)),
                         sharex=True, sharey=True)
if len(classes) == 1: axes = [axes]
cmap = plt.get_cmap("viridis")
for ax, c in zip(axes, classes):
    sub = ts[ts["trafficClass"] == c]
    for j, nid in enumerate(rx_nodes):
        ss = sub[sub["rx_node_id"] == nid]
        if ss.empty: continue
        agg = ss.groupby("t_bin_s")["prr_window"].mean()
        ax.plot(agg.index, agg.values,
                color=cmap(j / max(1, len(rx_nodes)-1)),
                lw=1.2, label=f"N{nid}", alpha=0.9)
    ax.set_ylabel(f"{c}\nPRR"); ax.set_ylim(0, 1.1)
axes[-1].set_xlabel("time [s]")
axes[0].legend(fontsize=7, ncol=4, loc="upper right")
fig.suptitle("Per-RX PRR over time — Adaptive MCS", y=1.01)
fig.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(classes), figsize=(3.0 * len(classes), 4.5), sharey=True)
if len(classes) == 1: axes = [axes]
for ax, c in zip(axes, classes):
    sub = ts[ts["trafficClass"] == c]
    if sub.empty: ax.set_visible(False); continue
    piv = sub.pivot_table(index="rx_node_id", columns="t_bin_s",
                          values="prr_window", aggfunc="mean")
    im = ax.imshow(piv.values, aspect="auto", vmin=0, vmax=1, cmap="RdYlGn",
                   origin="lower",
                   extent=[piv.columns.min(), piv.columns.max(),
                           piv.index.min()-0.5, piv.index.max()+0.5])
    ax.set_title(c, fontsize=10); ax.set_xlabel("time [s]")
    ax.set_yticks(sorted(ts["rx_node_id"].unique()))
axes[0].set_ylabel("RX node id")
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02).set_label("PRR (1-s window)")
fig.suptitle("PRR heatmap (RX × time) — Adaptive MCS", y=1.02)
plt.show()


## §5. Resource usage


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
ax = axes[0]
for c in classes:
    vals = ru[ru["trafficClass"] == c]["rbLen"].to_numpy()
    if len(vals) == 0: continue
    bins = np.arange(vals.min(), vals.max()+2) - 0.5
    ax.hist(vals, bins=bins, alpha=0.55, label=c, color=CLASS_COLORS[c],
            edgecolor="black", lw=0.4)
ax.set_xlabel("rbLen (PRBs/grant)"); ax.set_ylabel("count")
ax.set_title("PRB allocation distribution"); ax.legend(fontsize=8)

ax = axes[1]
grp = ru.groupby(["trafficClass", "seed"]).size().reset_index(name="n_grants")
means = grp.groupby("trafficClass")["n_grants"].agg(["mean", "std"]).reindex(classes)
ax.bar(range(len(classes)), means["mean"], yerr=means["std"],
       color=[CLASS_COLORS[c] for c in classes], alpha=0.8, capsize=3)
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20)
ax.set_ylabel("PSSCH grants per run"); ax.set_title("Channel occupancy")

ax = axes[2]
retx_share = []
for c in classes:
    sub = ru[ru["trafficClass"] == c]
    retx_share.append((sub["ndi"] == 0).sum() / max(len(sub), 1))
ax.bar(range(len(classes)), retx_share,
       color=[CLASS_COLORS[c] for c in classes], alpha=0.8)
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20)
ax.set_ylabel("retx / total TX"); ax.set_title("Retransmission share")
fig.suptitle("Resource usage by traffic class — Adaptive MCS", y=1.02)
fig.tight_layout(); plt.show()


## §6. HARQ statistics


In [ ]:
metrics = [("nack_rate", "PSFCH NACK rate"),
           ("retx_ratio", "MAC retx ratio"),
           ("phy_err_rate", "PHY decode-error rate"),
           ("timeout_count", "HARQ-process timeouts")]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (col, label) in zip(axes, metrics):
    agg = hq.groupby("trafficClass")[col].agg(["mean", "std"]).reindex(classes)
    ax.bar(range(len(classes)), agg["mean"], yerr=agg["std"],
           color=[CLASS_COLORS[c] for c in classes], alpha=0.85, capsize=3)
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=20)
    ax.set_ylabel(label); ax.set_title(label)
fig.suptitle("HARQ statistics by traffic class — Adaptive MCS", y=1.02)
fig.tight_layout(); plt.show()


## §6b. Adaptive-MCS behaviour

Controller trajectory, decision distribution, and NACK-rate window stats from
`adapt_log.csv`.


In [ ]:
if ad.empty:
    print("adapt_log.csv is empty — controller did not fire.")
else:
    fig, axes = plt.subplots(len(classes), 1, figsize=(11, 2.2*len(classes)),
                             sharex=True, sharey=True)
    if len(classes) == 1: axes = [axes]
    for ax, c in zip(axes, classes):
        sub = ad[ad["trafficClass"] == c]
        for seed, ss in sub.groupby("seed"):
            ax.plot(ss["t_s"], ss["new_mcs"], color="0.6", lw=0.5, alpha=0.4)
        mean = sub.groupby(sub["t_s"].round(1))["new_mcs"].mean()
        ax.plot(mean.index, mean.values, color=CLASS_COLORS[c], lw=2.2,
                label="mean across seeds")
        ax.axhline(14, color="k", lw=0.8, ls="--", alpha=0.5, label="MCS=14")
        ax.set_ylabel(f"{c}\nMCS"); ax.set_ylim(-0.5, 28.5)
        ax.legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("simulation time [s]")
    fig.suptitle("Adaptive-MCS controller trajectory per traffic class", y=1.01)
    fig.tight_layout(); plt.show()


In [ ]:
if ad.empty:
    print("adapt_log.csv is empty.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # Decision distribution
    ax = axes[0]
    dec = (ad.groupby(["trafficClass", "decision"]).size()
              .unstack(fill_value=0).reindex(classes))
    for col in ["up", "down", "same"]:
        if col not in dec.columns: dec[col] = 0
    dec = dec[["down", "same", "up"]]
    bottom = np.zeros(len(classes))
    colors = {"down": "#d7191c", "same": "#bbbbbb", "up": "#2ca02c"}
    for col in ["down", "same", "up"]:
        ax.bar(range(len(classes)), dec[col].values, bottom=bottom,
               label=col, color=colors[col], alpha=0.85)
        bottom += dec[col].values
    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=20)
    ax.set_ylabel("# evaluations")
    ax.set_title("Controller decisions per traffic class")
    ax.legend(loc="upper right", fontsize=9)

    # NACK-rate distribution
    ax = axes[1]
    sns.boxplot(data=ad, x="trafficClass", y="nack_rate",
                palette=CLASS_COLORS, ax=ax, showfliers=False, order=classes)
    ax.axhline(0.15, color="r", lw=0.8, ls="--", alpha=0.7, label="down_thr=0.15")
    ax.axhline(0.08, color="g", lw=0.8, ls="--", alpha=0.7, label="up_thr=0.08")
    ax.set_ylabel("NACK rate (per evaluation window)")
    ax.set_title("NACK-rate distribution per class")
    ax.legend(fontsize=8)
    for t in ax.get_xticklabels(): t.set_rotation(20)

    fig.suptitle("Adaptive controller diagnostics", y=1.02)
    fig.tight_layout(); plt.show()


## §7. Summary table


In [ ]:
def ci95(x):
    x = np.asarray(x); x = x[~np.isnan(x)]
    if len(x) == 0: return np.nan
    return 1.96 * np.nanstd(x) / np.sqrt(len(x))

summary = (rx.groupby("trafficClass")
             .agg(prr_mean=("prr","mean"),
                  prr_p5=("prr", lambda x: np.nanpercentile(x, 5)),
                  pir_p95_ms=("pir_ms", lambda x: np.nanpercentile(x, 95)),
                  tput_p5_kbps=("tput_kbps", lambda x: np.nanpercentile(x, 5)))
             .reindex(classes))
hq_agg = hq.groupby("trafficClass")[["nack_rate","retx_ratio"]].mean().reindex(classes)
summary = summary.join(hq_agg)
if not ad.empty:
    mcs_mean = ad.groupby("trafficClass")["new_mcs"].mean().reindex(classes)
    summary["mean_mcs"] = mcs_mean
else:
    summary["mean_mcs"] = np.nan
summary.round(3)


**Interpretation reminders**

- For the large-packet classes (Periodic3, Aperiodic2) the fixed-MCS=14 baseline
  achieved PRR=0% / ~15%.  Watch the §6b MCS trajectory: if the controller
  drops MCS sustainedly into the single digits, the §2 PRR violins for those
  classes should sit much higher than zero.
- For Periodic1 / Aperiodic1 (already ≥86% PRR at fixed MCS=14) the controller
  should mostly hold or slightly climb — adaptation should not gratuitously
  cost PRR on already-healthy flows.
